# 02 · Market intelligence

What Brazilian import data says about the category: implied import prices, how concentrated the origins are, and whether prices follow copper.

Data: Comex Stat monthly imports for NCM 8544.30.00 (harnesses), 8536.69.10 and 8536.69.90 (connectors), 2019 onward, and the World Bank Pink Sheet copper price. Load them with `python -m src.data.download_comexstat` and `python -m src.data.download_pinksheet`. The SQL views are in `sql/views.sql`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.db import DEFAULT_DB, connect, create_schema
from src.market import lag_correlation

con = connect(DEFAULT_DB)
create_schema(con)  # refreshes the views


def q(sql):
    return con.execute(sql).df()


NAMES = {
    "85443000": "8544.30.00 harnesses",
    "85366910": "8536.69.10 shielded connectors",
    "85366990": "8536.69.90 other connectors",
}

## Import value by year

In [ ]:
value = (
    q("""
    SELECT year(month) AS year, ncm, sum(usd_fob) / 1e6 AS usd_m
    FROM market_monthly GROUP BY 1, 2
""")
    .pivot(index="year", columns="ncm", values="usd_m")
    .rename(columns=NAMES)
)
ax = value.plot.bar(figsize=(9, 3.5), logy=True, rot=0)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8, frameon=False)
ax.set(xlabel="", ylabel="USD million (log scale)", title="Brazilian imports by year")
value.round(1)

## Implied import price

Total FOB value over total net weight, pooling all origins. Harnesses are reported in kilograms; connectors are counted in units, so they also get a price per unit. The last year is partial.

In [ ]:
implied = q("SELECT * FROM v_market_implied ORDER BY month")
implied["month"] = pd.to_datetime(implied["month"])
copper = q("SELECT month, copper_usd_t FROM copper_monthly WHERE month >= DATE '2019-01-01'")
copper = copper.set_index(pd.to_datetime(copper["month"]))["copper_usd_t"]

harness = implied[implied["ncm"] == "85443000"].set_index("month")["usd_per_kg"]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(harness, color="tab:blue", alpha=0.35, label="Harness USD/kg, monthly")
ax.plot(harness.rolling(12).mean(), color="tab:blue", label="Harness USD/kg, 12-month mean")
ax.set_ylabel("USD/kg")
ax2 = ax.twinx()
ax2.plot(copper / 1000, color="tab:orange", label="Copper USD/kg (Pink Sheet)")
ax2.set_ylabel("Copper USD/kg")
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=3, fontsize=8, frameon=False)
ax.set_title("Harness implied import price vs. copper");

In [ ]:
yearly = (
    implied.assign(year=implied["month"].dt.year)
    .groupby(["ncm", "year"])[["usd_fob", "kg_net", "qty_stat"]]
    .sum()
    .assign(
        usd_per_kg=lambda d: d["usd_fob"] / d["kg_net"],
        usd_per_unit=lambda d: d["usd_fob"] / d["qty_stat"],
    )[["usd_per_kg", "usd_per_unit"]]
    .rename(index=NAMES, level=0)
)
yearly.loc[yearly.index.get_level_values(0) == NAMES["85443000"], "usd_per_unit"] = None
yearly.round(2)

## Origin concentration

Herfindahl-Hirschman index of import value by country of origin, from 0 to 10,000. Above 2,500 is usually read as highly concentrated, 1,500 to 2,500 as moderately concentrated.

In [ ]:
hhi = q("SELECT * FROM v_market_hhi ORDER BY ncm, year")
ax = (
    hhi.pivot(index="year", columns="ncm", values="hhi")
    .rename(columns=NAMES)
    .plot(marker="o", figsize=(9, 3.5))
)
for cut in (1500, 2500):
    ax.axhline(cut, ls="--", lw=0.8, color="grey")
ax.set(xlabel="", ylabel="HHI", title="Origin concentration of imports")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8, frameon=False)
hhi.assign(ncm=hhi["ncm"].map(NAMES), top_share=hhi["top_share"].round(2), hhi=hhi["hhi"].round(0))

In [ ]:
q("""
    SELECT ncm, country, round(sum(usd_fob) / 1e6, 1) AS usd_m_2025,
           round(sum(usd_fob) / sum(sum(usd_fob)) OVER (PARTITION BY ncm), 3) AS share
    FROM market_monthly WHERE year(month) = 2025
    GROUP BY ncm, country
    QUALIFY row_number() OVER (PARTITION BY ncm ORDER BY sum(usd_fob) DESC) <= 5
    ORDER BY ncm, usd_m_2025 DESC
""").assign(ncm=lambda d: d["ncm"].map(NAMES))

## Does the import price follow copper?

Correlation between the monthly implied price and copper k months earlier, k = 0 to 6. Month-over-month log changes are the honest test; levels are shown for reference, but two series that both trend upward correlate in levels even without any link.

In [ ]:
rows = []
for ncm, g in implied.groupby("ncm"):
    price = g.set_index("month")["usd_per_kg"]
    for method in ("log_diff", "level"):
        r = lag_correlation(price, copper, max_lag=6, method=method)
        rows.append(r.assign(ncm=NAMES[ncm], method=method))
corr = pd.concat(rows)
table = corr.pivot_table(index=["ncm", "method"], columns="lag", values="corr").round(2)
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, method in zip(axes, ("log_diff", "level"), strict=True):
    corr[corr["method"] == method].pivot(index="lag", columns="ncm", values="corr").plot.bar(
        ax=ax, legend=False, rot=0
    )
    ax.axhline(0, color="black", lw=0.8)
    ax.set(title=f"Correlation with lagged copper ({method})", xlabel="lag (months)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=3,
    fontsize=8,
    frameon=False,
);

## Reading

- **Harnesses** rose from about USD 25.6/kg in 2019 to about USD 30/kg in 2025–2026 (the April 2020 spike is a low-volume pandemic month, not a price move). Origins are highly concentrated (HHI 2,400–3,500), with Paraguay alone at 46–58% of import value: a single-country dependency that belongs in the category risk score.
- **Shielded connectors (8536.69.10)** are even more concentrated, with China at about 60%. **Other connectors (8536.69.90)** are moderately concentrated and split between China, the United States and Germany.
- **Copper link:** month to month, implied prices barely move with copper (|r| below 0.2 at every lag). In levels, harnesses reach about 0.5 at a 3–4 month lag, which is consistent with lagged copper pass-through but also with a shared upward trend. Import unit values mix very different products under one NCM, so monthly noise from product mix hides the copper signal.
- **Implication for the should-cost:** calibrate on annual averages per family rather than monthly points, report the Comex Stat price as a range, and take the copper pass-through from the bottom-up cost tree instead of from this correlation.